In [1]:
import pandas as pd 

training_set = pd.read_csv('/home/s6moakba/InstructABSA/Dataset/SemEval14/Train/Restaurants_Train.csv')
training_set_15 = pd.read_csv('/home/s6moakba/InstructABSA/Dataset/SemEval15/Train/Restaurants_Train.csv')
training_set_16 = pd.read_csv('/home/s6moakba/InstructABSA/Dataset/SemEval16/Train/Restaurants_Train.csv')
test_set = pd.read_csv('/home/s6moakba/InstructABSA/Dataset/SemEval14/Test/Restaurants_Test.csv')
test_set_15 = pd.read_csv('/home/s6moakba/InstructABSA/Dataset/SemEval15/Test/Restaurants_Test.csv')
test_set_16 = pd.read_csv('/home/s6moakba/InstructABSA/Dataset/SemEval16/Test/Restaurants_Test.csv')
# yelp = pd.read_csv('/home/s6moakba/InstructABSA/Dataset/gen_data_dist_yelp_filtered.csv')
# lex_gen_filtered = pd.read_csv("/home/s6moakba/InstructABSA/Dataset/gen_lex.csv")

In [2]:
import pickle
with open('lex_terms_noun.pkl', 'rb') as file:
    lexicon_terms = pickle.load(file)

In [3]:
import pickle
with open('/home/s6moakba/Thesis/Filter_nouns/rest_nouns_filtered.pkl', 'rb') as file:
    yelp_terms = pickle.load(file)

In [4]:
lexicon_terms = {term.lower() for term in lexicon_terms}
yelp_terms = {term.lower() for term in yelp_terms}

In [5]:
validation_ratio = 0.1
num_samples = len(training_set)
num_validation_samples = int(validation_ratio * num_samples)

validation_set = training_set.sample(n=num_validation_samples, random_state=42)
training_set = training_set.drop(validation_set.index)

In [6]:
training_set_all = pd.concat([training_set, training_set_15, training_set_16])

In [7]:
test_set_all = pd.concat([test_set, test_set_15, test_set_16])

In [8]:
import ast
test_set['aspectTerms'] = test_set['aspectTerms'].apply(ast.literal_eval)
training_set['aspectTerms'] = training_set['aspectTerms'].apply(ast.literal_eval)
validation_set['aspectTerms'] = validation_set['aspectTerms'].apply(ast.literal_eval)
training_set_all['aspectTerms'] = training_set_all['aspectTerms'].apply(ast.literal_eval)
test_set_15['aspectTerms'] = test_set_15['aspectTerms'].apply(ast.literal_eval)
test_set_16['aspectTerms'] = test_set_16['aspectTerms'].apply(ast.literal_eval)
test_set_all['aspectTerms'] = test_set_all['aspectTerms'].apply(ast.literal_eval)
# yelp['aspectTerms'] = yelp['aspectTerms'].apply(ast.literal_eval)
# lex_gen_filtered['aspectTerms'] = lex_gen_filtered['aspectTerms'].apply(ast.literal_eval)


In [9]:
def get_unique_terms(df, column_name):
    terms = set()
    for row in df[column_name]:
        for item in row:
            terms.add(item['term'].lower())
    return terms

In [10]:
training_terms = get_unique_terms(training_set, 'aspectTerms')
validation_terms = get_unique_terms(validation_set, 'aspectTerms')
test_terms = get_unique_terms(test_set, 'aspectTerms')
training_terms_all = get_unique_terms(training_set_all, 'aspectTerms')
test_set_all = get_unique_terms(test_set_all, 'aspectTerms')

test_terms_16 = get_unique_terms(test_set_16, 'aspectTerms')
test_terms_15 = get_unique_terms(test_set_15, 'aspectTerms')
# lex_gen_terms = get_unique_terms(lex_gen_filtered, 'aspectTerms')
# yelp_terms = get_unique_terms(yelp, 'aspectTerms')

In [11]:
print('Training terms:', len(training_terms))
print('Validation terms:', len(validation_terms))
print('Test terms:', len(test_terms))

print('Lex gen terms:', len(lexicon_terms))
print('Yelp terms:', len(yelp_terms))

Training terms: 1120
Validation terms: 209
Test terms: 523
Lex gen terms: 857
Yelp terms: 1305


In [12]:
validation_terms_not_train = validation_terms - training_terms
validation_terms_not_train

{'apple tarte tatin',
 'apps',
 'architecture',
 'area',
 'assorted sashimi',
 'back waiters',
 'banana tempura',
 'basic dishes',
 'beverages',
 'blond wood decor',
 'bombay cosmopolitan',
 'bottle minimun',
 'brioche and lollies',
 'cafe',
 'captain',
 'ceviche mix (special)',
 'chicken tikka',
 'chicken with black bean sauce',
 'chicken with garlic sauce',
 'chocolate cake',
 'choice',
 'chu chu curry',
 'cigar bar',
 'cook',
 'crab dumplings',
 'cream cheese',
 'customers',
 'dals',
 'dance floor',
 'desert',
 'desserts with frog jelly',
 'dinner specials',
 'eat',
 'edamame pureed',
 'eggs benedict',
 'escargot',
 'food suggestions',
 'fresh mozzarella',
 'fromager',
 'garlic knots',
 'glass of beer',
 'gourmet food',
 'grapes',
 'half-price saturday night special',
 'hall',
 'hollondaise sauce',
 'hunan chicken',
 'indian fast food',
 'kamasutra',
 'ladies',
 'lamb sausages',
 'large whole shrimp',
 'lazy susans',
 'line',
 'lobster teriyaki',
 'mascarpone with chocolate chip',
 

In [12]:
test_terms_not_train = test_terms - training_terms
test_terms_not_train_15 = test_terms_15 - training_terms
test_terms_not_train_16 = test_terms_16 - training_terms
print(len(test_terms_not_train))
print(len(test_terms_not_train_15))
print(len(test_terms_not_train_16))

339
166
183


In [13]:
print(len(test_terms_16))
print(len(training_terms))

289
1120


In [14]:
target_terms = test_terms_not_train | test_terms_not_train_15 | test_terms_not_train_16 | validation_terms_not_train

In [15]:
availabe_terms = lexicon_terms | yelp_terms

In [16]:
remaining_terms = target_terms - availabe_terms

In [17]:
one_word_terms = set()
for term in remaining_terms:
    if len(term.split()) == 1:
        one_word_terms.add(term)

In [18]:
len(one_word_terms)

164

In [19]:
import random

# Calculate 60% of the length of availabe_terms
sample_size = int(0.75 * len(one_word_terms))

# Randomly sample 60% of the words
sampled_terms = random.sample(list(one_word_terms), sample_size)
len(sampled_terms)
sampled_terms = set(sampled_terms)

In [20]:
new_term_set = lexicon_terms | yelp_terms | sampled_terms  

In [56]:
print(len(test_terms_not_train & new_term_set) , len(test_terms_not_train))
print(len(test_terms_not_train_15 & new_term_set), len(test_terms_not_train_15))
print(len(test_terms_not_train_16 & new_term_set), len(test_terms_not_train_16))
print(len(validation_terms_not_train & new_term_set), len(validation_terms_not_train))

72 339
47 166
46 183
25 93


In [28]:
rest_new =  new_term_set - training_terms

In [29]:
import pickle

# Specify the file path to save the pickle file
pickle_file_path = 'res_gathered_noun_nt.pkl'

# Save the variable as a pickle file
with open(pickle_file_path, 'wb') as file:
    pickle.dump(new_term_set, file)

# GPT word analysis


In [13]:
gpt_terms = pd.read_csv('/home/s6moakba/gpt_res_nouns.csv')

In [14]:
gpt_terms = set(gpt_terms['word'].tolist())

In [17]:
common_terms = gpt_terms.intersection(test_terms_not_train)
print(common_terms)

{'happy hour', 'fried calamari', 'mole sauce', 'veggie burger'}


# use LEXICON

In [94]:
from nltk.corpus import wordnet as wn

def get_related_words_by_category(word_set, target_categories):
    # Step 1: Gather all related words without filtering by category
    all_related_words = set()
    
    for word in word_set:
        synsets = wn.synsets(word)  # Get synsets for each word
        
        for synset in synsets:
            # Add synonyms
            all_related_words.update(lemma.name().replace('_', ' ') for lemma in synset.lemmas())
            
            # Add hypernyms, allowing for multiple levels
            for hypernym in synset.hypernyms():
                all_related_words.update(lemma.name().replace('_', ' ') for lemma in hypernym.lemmas())
                    
            # Add hyponyms
            for hyponym in synset.hyponyms():
                all_related_words.update(lemma.name().replace('_', ' ') for lemma in hyponym.lemmas())
                
            # Add meronyms and holonyms
            for meronym in synset.part_meronyms():
                all_related_words.update(lemma.name().replace('_', ' ') for lemma in meronym.lemmas())
            for holonym in synset.member_holonyms():
                all_related_words.update(lemma.name().replace('_', ' ') for lemma in holonym.lemmas())
                
            # Add entailments
            for entailment in synset.entailments():
                all_related_words.update(lemma.name().replace('_', ' ') for lemma in entailment.lemmas())
            
            # Add derivationally related forms
            all_related_words.update(lemma.name().replace('_', ' ') for lemma in synset.lemmas() if lemma.derivationally_related_forms())
    
    # Step 2: Filter the gathered words by category
    filtered_related_words = set()
    category_synsets = {wn.synsets(category)[0] for category in target_categories if wn.synsets(category)}

    for related_word in all_related_words:
        related_synsets = wn.synsets(related_word)
        
        # Check if any synset of the related word has a hypernym path that matches target categories
        for synset in related_synsets:
            if any(category_synset in synset.hypernym_paths()[0] for category_synset in category_synsets):
                filtered_related_words.add(related_word)
                break  # Add word once, then exit loop for efficiency
    
    return filtered_related_words

# Expanded target categories for broader filtering
target_categories = [
    "food", "drink", "cuisine", "ingredient", "meal", "course", "dish", "snack",
    "worker", "employee", "cook", "chef", "waiter", "furniture", "utensil",
    "appliance", "room", "music", "decor", "service", "menu", "reservation", 
    "establishment", "preparation", "serving", "portion", "order"
]

general_target_categories = [
    "food", "drink", "cuisine", "dish", "meal", "course", "snack", "ingredient",  # Food-related
    "cooking", "preparation", "garnish", "sauce", "flavor", "spice",              # Preparation & Flavor
    "service", "staff", "waiter", "server", "chef", "bartender", "cook", # Staff roles
    "utensil", "equipment", "furniture",                              # Equipment & Furniture
     "hall", "seating", "counter",      # Restaurant space & seating
    "decor", "atmosphere", "ambiance", "lighting",                       # Ambiance & Decor
    "menu", "special", "buffet", "price", "reservation",               # Dining logistics & offerings
                      # Entertainment
    "restaurant", "bar", "pub", "cafe", "bistro", "establishment"                 # Types of establishments
]



In [95]:
lexicon_terms_1 = get_related_words_by_category(training_terms_all, general_target_categories)

In [85]:
def find_root (word, target_categories):
    synsets = wn.synsets(word)
    for synset in synsets:
        for category in target_categories:
            if any(category_synset in synset.hypernym_paths()[0] for category_synset in wn.synsets(category)):
                print(category) 
        else:
            print('not found')

In [105]:
find_root('borage', general_target_categories)

not found
food
ingredient
not found


In [97]:
len(lexicon_terms_1)

870

In [98]:
len(lexicon_terms_1 & target_terms)

31

In [104]:
import random
random_item = random.sample(list(lexicon_terms_1), 10)
random_item

['stuffing',
 'biriani',
 'flavouring',
 'ale',
 'croquette',
 'Burger',
 'brewage',
 'beefburger',
 'borage',
 'lamp']

In [106]:
import pickle

# Specify the file path to save the pickle file
pickle_file_path = 'lex_terms_noun.pkl'

# Save the variable as a pickle file
with open(pickle_file_path, 'wb') as file:
    pickle.dump(lexicon_terms_1, file)